In [1]:
!pip install -qU langchain langchain-google-genai langchain-community langchain-text-splitters langgraph pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [4]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# Load the API key securely from Google Colab Secrets
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# 1. Load the PDF document
loader = PyPDFLoader("depshield_proposal.pdf")
docs = loader.load()

# Store 1-indexed page numbers in metadata for accurate citations
for doc in docs:
    doc.metadata["pdf_page"] = doc.metadata.get("page", 0) + 1

# 2. Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splits = text_splitter.split_documents(docs)

# 3. Generate embeddings and index into InMemoryVectorStore
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.environ["GEMINI_API_KEY"],
)

vector_store = InMemoryVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
)

print(f"Success! Loaded {len(docs)} pages and indexed {len(splits)} chunks.")

Success! Loaded 13 pages and indexed 33 chunks.


In [13]:
from datetime import datetime
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver

# Tool 1: RAG Retriever Tool with Page Citations
@tool
def search_proposal(query: str) -> str:
    """Search the DepShield AI FYP proposal document and return relevant excerpts with page numbers."""
    results = vector_store.similarity_search(query, k=4)
    if not results:
        return "No relevant information found in the document."

    formatted_chunks = []
    for doc in results:
        page_num = doc.metadata.get("pdf_page", "Unknown")
        formatted_chunks.append(f"[Page {page_num}]: {doc.page_content}")

    return "\n\n---\n\n".join(formatted_chunks)

# Tool 2: Extra Custom Tool tailored to the FYP Proposal topic
@tool
def save_advisor_note(section_title: str, note_content: str) -> str:
    """Save an important summary, action item, or advisor meeting note from the DepShield AI proposal into a markdown file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    entry = f"## {section_title} ({timestamp})\n{note_content}\n\n---\n\n"
    with open("advisor_notes.md", "a", encoding="utf-8") as f:
        f.write(entry)
    return f"Successfully saved note under '{section_title}' to advisor_notes.md."

# Initialize fast Gemini Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    timeout=30,
    max_retries=2,
    google_api_key=os.environ["GEMINI_API_KEY"],
)

# Initialize Conversation Memory
memory = InMemorySaver()

system_prompt = """You are an AI Research & FYP Advisor Assistant for the 'DepShield AI' project proposal.
Rules:
1. Always use the `search_proposal` tool to answer questions about the DepShield AI proposal.
2. Cite the exact PDF page number in parentheses for every fact you state, e.g., (page 5).
3. If the answer is not explicitly present in the document, clearly state that the document does not mention it—do NOT guess or make up facts.
4. When the user asks to save a note, summary, or milestone for advisor meetings, use the `save_advisor_note` tool.
5. Use conversation history to understand follow-up questions like "what about the second one?" or "explain that further"."""

agent = create_agent(
    model=llm,
    tools=[search_proposal, save_advisor_note],
    system_prompt=system_prompt,
    checkpointer=memory,
)

def ask_agent(question: str, thread_id: str = "depshield_session_1"):
    config = {"configurable": {"thread_id": thread_id}}
    response = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    raw_content = response["messages"][-1].content

    # Extract clean text from structured Gemini response blocks
    if isinstance(raw_content, list):
        answer = "\n".join(
            block.get("text", "") for block in raw_content if isinstance(block, dict) and "text" in block
        )
    else:
        answer = str(raw_content)

    print(f"User: {question}\n")
    print(f"Agent: {answer}\n")
    print("=" * 70)

print("Agent updated and initialized successfully!")

Agent updated and initialized successfully!


In [14]:
# Demo 1: Conversation Memory (Follow-up question test)
ask_agent("Who are the group members for the DepShield AI project and what are their CGPAs?", thread_id="memory_demo")

# Follow-up question relying on conversation memory ("the second one")
ask_agent("What is the registration number and contact number of the second one?", thread_id="memory_demo")

# Demo 2: Using the Extra Custom Tool (save_advisor_note)
ask_agent("What is the tech stack planned for Frontend, Backend, and Database? Please save a concise summary of it to my advisor notes under the title 'Planned Tech Stack'.", thread_id="memory_demo")

User: Who are the group members for the DepShield AI project and what are their CGPAs?

Agent: Based on the DepShield AI proposal, the group members and their CGPAs are:

1. **Tasmiya Iram Syeda** (Registration #: L1F23BSCS0738) – CGPA: **3.83** (page 1)
2. **Haram Tariq** (Registration #: L1F23BSCS0691) – CGPA: **3.01** (page 1)
3. **Hamna Sarfaraz** (Registration #: L1F23BSCS0696) – CGPA: **2.83** (page 1)

User: What is the registration number and contact number of the second one?

Agent: For the second group member, **Haram Tariq**:
* **Registration Number:** L1F23BSCS0691 (page 1)
* **Contact Number:** 0327-1040738 (page 1)

User: What is the tech stack planned for Frontend, Backend, and Database? Please save a concise summary of it to my advisor notes under the title 'Planned Tech Stack'.

Agent: Based on the proposal (page 8), the planned tech stack is:

* **Frontend:** React.js, Tailwind CSS (to build the user interface)
* **Backend:** Node.js with Express.js or Python FastAPI 

In [15]:
import time

test_questions = [
    # 8 Questions answerable from the PDF
    "1. Who is the Project Advisor for DepShield AI and how often must supervisory meetings be held?",
    "2. Which dependency manifest files does DepShield AI scan?",
    "3. According to the Gap Analysis table, how does DepShield AI compare to GitHub Dependabot in checking whether a dependency is actually used in project code?",
    "4. What are the 5 factors checked for each dependency before calculating the context-aware risk score in the system flowchart?",
    "5. What are the exact KPIs mentioned for testing and user feedback in Section 8?",
    "6. According to the 24-week Gantt chart, during which weeks is 'Vulnerability lookup: OSV/NVD/GitHub' scheduled?",
    "7. In the 16-week project schedule table, what is the deliverable for Weeks 11-12?",
    "8. Which research paper is cited as reference [8] in the bibliography and what methodology does it introduce?",

    # 2 Questions NOT in the PDF (to test hallucination resistance)
    "9. What is the exact mathematical formula and weights used to calculate the hybrid risk score in DepShield AI?",
    "10. What is the estimated cloud hosting budget in PKR for deploying DepShield AI on Vercel and Render?"
]

for idx, q in enumerate(test_questions, start=1):
    # Use a separate thread_id for each test question so every test is independent
    ask_agent(q, thread_id=f"test_q_{idx}")
    time.sleep(2)  # Short pause to avoid free-tier rate limits

User: 1. Who is the Project Advisor for DepShield AI and how often must supervisory meetings be held?

Agent: Based on the DepShield AI proposal:

* **Project Advisor:** Muhammad Zulkifl Hasan (page 2).
* **Supervisory Meetings Frequency:** A group is required to hold at least two meetings per month and maintain meeting minutes (page 2). (The project plan also notes that the advisor monitors progress through biweekly meetings on page 11).

User: 2. Which dependency manifest files does DepShield AI scan?

Agent: DepShield AI scans dependency files such as **package.json**, **package-lock.json**, and **requirements.txt** (page 5).

User: 3. According to the Gap Analysis table, how does DepShield AI compare to GitHub Dependabot in checking whether a dependency is actually used in project code?

Agent: According to the Gap Analysis table in the DepShield AI proposal, GitHub Dependabot has **"Limited"** support for checking whether a dependency is actually used in the project code, whereas 